<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Machine Learning Foundations and Data Preparation
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

The preparation notebook taught the methods; today is about the decisions, and ends with you building a feature table for a model. Four kinds of activity are marked with **»**:

* **» Predict** — *before* running, decide what the output will be, then run to check. Only used where you have already met the code.
* **» Investigate** — run a working cell, then answer a question about it, from the output or by changing one thing.
* **» Modify** — change a working cell so it answers the next question.
* **» Make** — write the code yourself. The pattern is always somewhere above. **Hint** blocks fold open if you need them: try first.

This follows the PRIMM pattern used in programming teaching: Predict, Run, Investigate, Modify, Make.

Part way through, an **Independent practice** slide hands over to you for a long stretch of self-paced work; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: the imports, the `query` function and every table you load exist in every cell after them, so the cells are meant to be run in order. After a restart, or if a cell complains that `df_stays` is not defined, use **Runtime → Run before** to catch up.

**MIMIC-IV outputs are not in the published slides or notes.** The data use agreement forbids sharing the data, so you see them only when you run the cells yourself. For the same reason, never paste MIMIC rows into an AI assistant or share them; asking about your code is fine.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will have turned raw ICU tables into a feature table that a model could learn from, and be able to say what each column assumes.

Sessions 1 to 3 covered loading data into Python and asking questions of it. The preparation notebook added the methods for exploring, cleaning and fixing a table. Today uses them: what a machine learning model needs, which decisions the methods leave to you, and then a long stretch of practice building a feature table from the course MIMIC views.

### Before we start
1. Open this notebook in Colab: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/04_ML_Foundations_and_Data_Preparation.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Run the **setup** cells in section 1 now: they load the tables for the whole session and take a minute or two.

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Describe** what a machine learning model is, and **explain** why a model that fits its training data perfectly can still be a bad model.
2. **Explain** what a feature table is, and **identify** which of its columns are known at the time a prediction would be made.
3. **Choose** between dropping, filling and keeping missing, extreme and inconsistent values, and **justify** the choice from how the data were produced.
4. **Encode**, **engineer** and **scale** features for a model, and **explain** what each transformation assumes.
5. **Build** a feature table from several clinical tables, aligned to one row per prediction and one prediction time.

## Session outline
1. Setup, and a readiness check on the preparation notebook
2. Machine learning foundations
3. Exploring data for a model
4. Five questions for every feature
5. Known at prediction time?
6. How does it get into the row?
7. Is it complete and correct?
8. Does it carry what you know?
9. Is it a number the model can work with?
10. Independent practice: a feature table for in-hospital death
11. Closing

# 1. Setup
The same course views as the preparation notebook, plus three new ones over the **ICU stays** in the cohort: `icustays`, and two tables derived by the MIMIC team, `first_day_vitalsign` and `first_day_lab`, which summarise the first 24 hours of each ICU stay. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**.

Loading takes a minute or two, so **run this once**.

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import matplotlib.pyplot as plt
import pandas_gbq

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.chartevents, keeping itemid 220045 (heart rate)
df_hr = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.chartevents_hr`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_hr))

**`df_stays`** is the table for the whole session: **one row per ICU stay**, with the admission and the patient it belongs to, and the first-day vital signs. Two columns are added: `died`, 1 if the patient died in hospital, and `los_days`, the length of the hospital stay in days. The temperature columns come back from BigQuery as `Decimal` objects rather than floats, a type pandas cannot do arithmetic on, so they are converted here. As the preparation notebook said, check a column's type before you use it.

In [ ]:
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
stay_length = df_stays['dischtime'] - df_stays['admittime']
df_stays['los_days'] = stay_length.dt.total_seconds() / (60 * 60 * 24)

# The temperature columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)

## Readiness check
Four situations from the preparation notebook. **On your own, type a one-line answer to each in the cell below.**

1. A length-of-stay column has a **mean of 10 days and a median of 7**. Which do you report as the typical stay, and what does the gap tell you about the shape of the data?
2. `df_admissions.dropna()` left **only patients who died**. Why? What would a model trained on the rows it kept learn?
3. On raw heart rates the z-score flagged **2** readings; after masking values over 250 bpm it flagged **5,100**. Which run do you trust, and why did the first one miss so much?
4. `pd.to_numeric(df['Weight'], errors='coerce')` ran without an error and kept **2 of 10** weights. What is wrong with the two it kept?

Then, in your pair, **compare your answers** and agree on one for each question. We will take a show of hands.

In [ ]:
#@title Your answers (nothing is checked: they are for the comparison with your neighbour)
answer_1 = ""  #@param {type:"string"}
answer_2 = ""  #@param {type:"string"}
answer_3 = ""  #@param {type:"string"}
answer_4 = ""  #@param {type:"string"}

## What we were checking
1. **The median.** A mean well above the median means a long right tail: a few very long stays pull the mean up. The tail is real, but it is not typical.
2. **Why a value is missing.** `deathtime` is empty for everyone who survived, so "complete rows only" kept the deaths. A model trained on them would learn that everyone dies. Missingness can carry information; find out why before you drop or fill.
3. **The second run.** One reading of five million inflates the standard deviation so much that nothing else looks extreme. A statistic computed from the errors cannot find the errors: apply the clinical limits first.
4. **No unit.** `130` and `165.3` were kept because they had no unit to break the conversion, so you cannot tell kilograms from pounds. A conversion that runs without error is not necessarily correct.

Today is about decisions like these: when each method helps, and when it misleads.

# 2. Machine learning foundations
*   The next four sessions apply machine learning to clinical data
*   The definition we follow in this course:

> *"[when] a computer observes some data, builds a model based on the data, and uses the model as both a hypothesis about the world and a piece of software that can solve problems"*
>
> Artificial Intelligence: A Modern Approach, Fourth Edition. Russell and Norvig. Pearson Education Limited, 2022

![ML illustration](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-03_ML-Foundations.svg)

## What machine learning is not
*   Definitions vary, but machine learning is a subset of artificial intelligence
*   It does not include AI approaches that do not learn from data:
    *   Search algorithms
    *   Rule-based agents
    *   Logical agents
*   A clinical early warning score with fixed thresholds is a rule-based system; a model that learned the thresholds from outcomes is machine learning

## Modelling data
*   Building models of data is centuries old; machine learning is one way of doing it
*   A model lets you understand and predict how variables relate to each other
*   A model is simpler than the process it represents

For a physical process:

$$y = f(X, Z)$$

*   $y$ is the output (the "dependent variable"), e.g. length of stay
*   $f()$ is the physical process
*   $X$ are the observed inputs, e.g. age, admission type, first-day heart rate
*   $Z$ are the unobserved inputs, e.g. frailty, what the surgeon saw

## Why use a model
Forty made-up patients: an input $X$, say age, and an outcome $y$, say length of stay. Three ways to use them.

*   **The points** answer a question about a patient only if that patient is in the data
*   **Aggregation**, Session 2's `groupby` and box plots by band, gives a typical value per band: nothing between the bands, and the sparse bands are unreliable
*   **A model** gives a value at any $X$, uses every point, can take many inputs at once, and can be written down, tested and reused

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Made-up patients: an outcome that curves with X, plus what we do not measure (Z)
rng = np.random.default_rng(119)
def process(x):
    return 4 + 2.5 * np.sin((x - 20) / 18)
X = np.sort(rng.uniform(22, 88, 40))
y = process(X) + rng.normal(0, 0.6, len(X))
grid = np.linspace(22, 88, 300)

# A gentle curve fitted to all forty points
model = np.polynomial.Polynomial.fit(X, y, 3)

def tidy(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylim(0.5, 8.5)
    ax.set_xlabel('$X$')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
axes[0].scatter(X, y, color='#444444', s=24, zorder=3)
axes[0].set_title('The data')

band_edges = [22, 40, 55, 70, 88]
bands = [y[(X >= low) & (X < high)] for low, high in zip(band_edges[:-1], band_edges[1:])]
centres = [(low + high) / 2 for low, high in zip(band_edges[:-1], band_edges[1:])]
axes[1].boxplot(bands, positions=centres, widths=10, manage_ticks=False)
axes[1].set_title('Aggregated by band')

axes[2].scatter(X, y, color='#444444', s=24, zorder=3)
axes[2].plot(grid, model(grid), color='#0000cd', linewidth=2)
axes[2].set_title('A model')
for ax in axes:
    tidy(ax)
axes[0].set_ylabel('$y$')
plt.tight_layout()
plt.show()

### A model of the process
$$y = \hat{f}(X) + \epsilon$$

*   $\hat{f}()$ is the model's approximation of the process, built from $X$ alone
*   $\epsilon$ is the error: the model's own shortcomings, plus everything in $Z$

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(X, y, color='#444444', s=22, zorder=3, label='$y = f(X, Z)$: what happened to each patient')
ax.plot(grid, model(grid), color='#0000cd', linewidth=2.2, label=r'$\hat{f}(X)$: the model, fitted to the points')

# One mid-range patient, to show the error
middle = (X > 30) & (X < 80)
i = np.argmax(np.abs(y - model(X)) * middle)
ax.plot([X[i], X[i]], [model(X[i]), y[i]], color='#c0392b', linewidth=2, zorder=4)
ax.annotate(r'$\epsilon$: the error for this patient', xy=(X[i], (model(X[i]) + y[i]) / 2), xytext=(X[i] - 30, y[i] - 1.8),
            color='#c0392b', fontsize=11, arrowprops=dict(arrowstyle='-', color='#c0392b', linewidth=1))

ax.set_xlabel('$X$: an observed input, e.g. age')
ax.set_ylabel('$y$: the outcome, e.g. length of stay')
ax.set_xticks([])
ax.set_yticks([])
ax.set_ylim(0.5, 8.5)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
ax.legend(loc='upper right', frameon=False, fontsize=10)
plt.tight_layout()
plt.show()

## What makes a good model
For $y = \hat{f}(X) + \epsilon$, a good model is one with a small error $\epsilon$. Three models fitted to twelve of the forty patients, with the average error of each.

**Which of the three is the best model, and why?**

In [ ]:
# Twelve of the forty patients to fit on; the other twenty-eight are kept back
train = np.sort(rng.choice(len(X), 12, replace=False))
new = np.setdiff1d(np.arange(len(X)), train)
X_train, y_train, X_new, y_new = X[train], y[train], X[new], y[new]

# Three models: a straight line, a gentle curve, and a polynomial through every training point
degrees = {'Model A': 1, 'Model B': 3, 'Model C': 9}
fits = {name: np.polynomial.Polynomial.fit(X_train, y_train, deg) for name, deg in degrees.items()}

def error(fit, X_, y_):
    return np.mean(np.abs(fit(X_) - y_))

fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, (name, fit) in zip(axes, fits.items()):
    ax.plot(grid, fit(grid), color='#0000cd', linewidth=2)
    ax.scatter(X_train, y_train, color='#444444', s=24, zorder=3)
    ax.set_title(f'{name}: training error {error(fit, X_train, y_train):.2f}', fontsize=11)
    tidy(ax)
axes[0].set_ylabel('training data')
plt.tight_layout()
plt.show()

### Overfitting and underfitting
*   What matters is the error on **future, unseen** data: the same three models, on the twenty-eight patients they were not fitted to
*   Model C fitted its training data almost perfectly and does as badly as the straight line on new patients: it is **overfitted**, having learnt the noise as well as the pattern
*   Model A cannot bend, so it is wrong on both: **underfitted**
*   Much of the next three sessions is about finding Model B. Today's part: a table with hundreds of columns and a few thousand rows invites overfitting, so every column needs a reason to be there

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, (name, fit) in zip(axes, fits.items()):
    ax.plot(grid, fit(grid), color='#0000cd', linewidth=2)
    ax.scatter(X_new, y_new, color='#c0392b', s=24, zorder=3)
    ax.set_title(f'{name}: training {error(fit, X_train, y_train):.2f}, new patients {error(fit, X_new, y_new):.2f}', fontsize=10)
    tidy(ax)
axes[0].set_ylabel('new patients', color='#c0392b')
plt.tight_layout()
plt.show()

## What a model needs: a feature table
*   **One row per thing you predict about**: a patient, an admission, an ICU stay, an hour
*   **Numeric columns**: the model is arithmetic, so every feature has to be a number
*   **A label column**: what you want to predict, known for the training rows. This course says **features** and **labels**; the other names you will meet are in the appendix
*   **Only what is known at prediction time**: a column that leaks the future gives a model that works on the training table and fails in the clinic

![Illustration of target structure for a data set](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-03_Feature-Table.svg)

Data preparation is getting from clinical tables to this shape.

## Numbers that are true and misleading
Four datasets, A to D, each of 142 pairs of `x` and `y`. The summary statistics you learnt in the preparation notebook, for each.

In [ ]:
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"
df_dozen = pd.read_csv(DATA + "datasaurus_dozen.csv")

# Four of the thirteen datasets in the file, relabelled A to D
labels = {'circle': 'A', 'star': 'B', 'x_shape': 'C', 'dino': 'D'}
df_four = df_dozen[df_dozen['dataset'].isin(labels)].copy()
df_four['dataset'] = df_four['dataset'].map(labels)

summary = df_four.groupby('dataset')[['x', 'y']].agg(['mean', 'std'])
summary.round(1)

The correlation between `x` and `y`, for each:

In [ ]:
for name in ['A', 'B', 'C', 'D']:
    one = df_four[df_four['dataset'] == name]
    print(name, round(one['x'].corr(one['y']), 2))

**» Predict** — Same means, same spreads, same correlation. **Are the four datasets the same? Or Similar?**

The four datasets, plotted:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, name in zip(axes, ['A', 'B', 'C', 'D']):
    one = df_four[df_four['dataset'] == name]
    one.plot.scatter(x='x', y='y', s=8, ax=ax, title=name)
plt.show()

## Why explore before you model
*   Every number on the previous slides was **true**, and every one was **misleading**
*   Summary statistics compress a dataset to a few numbers, and the compression hides things
*   **Numbers and pictures each hide what the other shows, so use both**
*   **Exploratory data analysis (EDA)** is looking at the data before you model it, going round a loop: **look**, **ask**, **answer**, **notice**, ask again
*   For a model, EDA has three jobs: check the data are what you think they are, find what needs cleaning, and see which features might carry information about the label

*Datasets from the Datasaurus Dozen, Matejka and Fitzmaurice (2017), CC BY 4.0.*

# 3. Exploring data for a model
The question for today's table: **which patients admitted to the ICU die in hospital?** The label is `died`. Before any feature, look at the label itself.

In [ ]:
df_stays['died'].value_counts(normalize=True)

**» Investigate** — About one stay in eight ends in death. **A model that predicts "survives" for everyone: what proportion of stays does it get right?** That number is the bar any real model has to beat, and Session 5 comes back to it.

## Does a feature separate the groups?
A grouped box plot puts a candidate feature against the label: the maximum heart rate in the first ICU day, for stays that ended in death and stays that did not.

In [ ]:
df_stays.boxplot(column='heart_rate_max', by='died')
plt.suptitle('')
plt.title('Maximum heart rate in the first ICU day, by in-hospital death')
plt.xlabel('Died in hospital')
plt.ylabel('Heart rate (bpm)')
plt.show()

**» Investigate** — The medians differ by about 13 bpm, with a lot of overlap. The next cell plots the same statistic over the **whole stay** instead of the first day, from the heart rate readings loaded in the setup. The separation is clearer. Is it a better feature? Section 5 has the answer.

A statistical test would call both differences significant, and that is not the question: with 9,600 rows almost any difference is. What matters for a feature is how far the groups separate, and whether the column is known at prediction time. The tests are in the appendix.

In [ ]:
plausible = df_hr[(df_hr['valuenum'] >= 20) & (df_hr['valuenum'] <= 250)]
hr_max_stay = plausible.groupby('hadm_id')['valuenum'].max()
df_stays['hr_max_stay'] = df_stays['hadm_id'].map(hr_max_stay)

df_stays.boxplot(column='hr_max_stay', by='died')
plt.suptitle('')
plt.title('Maximum heart rate over the whole stay, by in-hospital death')
plt.xlabel('Died in hospital')
plt.ylabel('Heart rate (bpm)')
plt.show()

# 4. Five questions for every feature
*   A model needs **one table**: every feature and the label, one row per prediction
*   Clinical data arrive in many: patients, admissions, stays, measurements, diagnoses, each with its own granularity and timing
*   Session 2's `merge` and `concat` do the joining; the work is deciding **what one row is** and **when the prediction is made**

![Illustration of target structure for a data set](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-03_Feature-Table.svg)

## The five questions
Every column in the feature table has to pass all five. One section each, and the practice asks them of each feature you build.

5. **Known at prediction time?** If not, it is leakage, however well it separates the groups.
6. **How does it get into the row?** One row per prediction: from which table, over which window, aggregated how.
7. **Is it complete and correct?** Missing for a reason, an error, a unit that changed.
8. **Does it carry what you know?** A derived quantity or a clinical threshold the raw columns do not hold.
9. **Is it a number the model can work with?** Text has to become numbers, and the numbers have to be on a scale the model can use.

# 5. Known at prediction time?
Today's model predicts in-hospital death **24 hours after ICU admission**. Everything in the row has to be known then. Our sources:

| Source | Known at 24 hours? |
| :-- | :-- |
| `patients`: gender, anchor age | Yes |
| `admissions`: admission time, type, race | Yes |
| `first_day_vitalsign`: first 24 hours of the ICU stay | Yes, just |
| `diagnoses_icd`: ICD codes for the admission | **No.** Coded after discharge, for billing |
| `admissions`: `hospital_expire_flag`, `dischtime`, `los` | **No.** This is the label, or determined by it |
| `patients`: `dod` | **No.** Death up to a year after the last discharge |

A column that is not known at prediction time is **leakage**: the model looks good on the table and fails in use. Diagnosis codes are the subtle case: a chronic diagnosis is real history, but the *code* for this admission did not exist at 24 hours.

# 6. How does it get into the row?
A feature table has one row per prediction, and clinical data arrive at every other grain: many readings an hour, one lab result a day, one diagnosis an admission. Each feature has to be aligned to the row and aggregated into it.

## Aligning in time
*   Heart rate is charted hourly, labs arrive when someone orders them, a diagnosis is coded after discharge
*   To put them in one row they have to be aligned to a **window** ending at the prediction time, and aggregated within it
*   **If you were building a risk score from hourly heart rate and ad hoc lab results, what would you do with each?**

## Which window?
Heart rate is charted many times an hour. A feature table has one row per stay, so the readings have to be **aggregated**: a mean, a maximum, a count. The window matters as much as the statistic. Two means of the same readings, per admission: the whole hospital stay, and the first 24 hours after hospital admission.

In [ ]:
df_timed = df_hr.merge(df_admissions[['hadm_id', 'admittime']], on='hadm_id')
time_since_admission = df_timed['charttime'] - df_timed['admittime']
df_timed['hours'] = time_since_admission.dt.total_seconds() / (60 * 60)

hr_whole_stay = df_timed.groupby('hadm_id')['valuenum'].mean()
df_first24h = df_timed[(df_timed['hours'] >= 0) & (df_timed['hours'] <= 24)]
hr_first_24h = df_first24h.groupby('hadm_id')['valuenum'].mean()

print(len(df_admissions) - len(hr_whole_stay), 'admissions with no heart rate over the whole stay')
print(len(df_admissions) - len(hr_first_24h), 'admissions with no heart rate in the first 24 hours')

**» Investigate** — The whole-stay mean has no missing values; the 24-hour mean is missing for about 1,700 admissions. **Which of the two could a model use to make a prediction 24 hours after admission?** Why is the complete column a problem?

`first_day_vitalsign`, in `df_stays`, uses a third window: the first 24 hours of the **ICU** stay. Only 10 stays have no heart rate in it. Why would the ICU window be so much more complete than the hospital one?

# 7. Is it complete and correct?
The preparation notebook gave you the methods: `isna`, `dropna`, `fillna`, `astype`, limits, z-scores, the IQR and `mask`. Each leaves a decision to you, and the decision depends on **how the data were produced**.

## Why is it missing?
Five columns of `df_stays`, and how many stays have no value in each.

In [ ]:
df_stays[['deathtime', 'dod', 'marital_status', 'heart_rate_mean', 'temperature_mean']].isna().sum()

**» Investigate** — **For each column, say why a value would be missing.** One of them is missing because of something that happened to the patient, one because of something that did not happen, one because nobody recorded it, and two because nobody measured it. Which is which, and which could you reasonably fill with a median?

Missing values are rarely spread evenly. `temperature_missing` is `True` where no temperature was recorded in the first day; here is how often that happens in each outcome group.

In [ ]:
df_stays['temperature_missing'] = df_stays['temperature_mean'].isna()
df_stays.groupby('died')['temperature_missing'].mean().round(3)

## Choosing how to fill
The choice of imputation depends on:

*   **Domain knowledge:** what the value represents, and why it is missing
*   **Data type:** numerical or categorical
*   **Amount of data:** small data limit the complex methods
*   **Amount missing:** a simple fill for a large fraction of a column invents a lot of data
*   **The model downstream:** some models cope with missing values, most do not

Two rules:

1. **Missing can be a feature.** Where the gap carries information (no temperature taken; no `deathtime`), keep a `_missing` column alongside the filled one, or encode the gap as a category.
2. **Imputation learns from the data.** A mean or a median is computed from rows; so is any method that predicts the missing value from the other columns. Session 5 splits the data into training and test sets, and the filler must be computed on the training rows only, or the test set is no longer unseen.

More methods, with code, are in the appendix.

## Error or extreme?
The first-day tables were derived by the MIMIC team, but derived does not mean clean. The maximum blood glucose in the first day, across the cohort:

In [ ]:
df_stays['glucose_max'].describe().round(1)

**» Investigate** — The largest value is not a blood glucose. The next cell lists every value above 1,000. **Which of them are errors, which could be real, and where would you put the limit?** Would a z-score or the IQR have made that decision for you?

In [ ]:
high_glucose = df_stays[df_stays['glucose_max'] > 1000]['glucose_max']
high_glucose.sort_values()

As in the preparation notebook: **clinical limits first, statistics second.** Statistics describe the data you keep; they cannot tell you what to remove.

# 8. Does it carry what you know?
The raw columns are what the hospital happened to record. **Feature engineering** adds what you know: a quantity the model would struggle to learn from the raw columns, or a clinical threshold it could not know about.

## Feature engineering
*   **Feature engineering** uses domain knowledge to make new features from the ones you have
*   Height and weight give BMI; systolic and diastolic pressure give pulse pressure; heart rate and blood pressure give a shock index
*   A model could, in principle, learn these from the raw columns. **Why make them anyway?**

### Why engineer features
*   Puts knowledge into the table that the training set may not contain
*   Lets a **simpler model** do the job, which is easier to explain and less likely to overfit
*   Makes the features mean something to the people who use the model
*   Reduces the number of columns, when the engineered feature replaces several

This is where your clinical knowledge matters. The three examples that follow are all from today's table.

### Derived features: age at admission
MIMIC gives no date of birth. `anchor_age` is the patient's age in `anchor_year`, and `admittime` is shifted by the same offset as `anchor_year`, so the age at admission is:

In [ ]:
df_stays['age'] = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']

print('Stays where age differs from anchor_age:', (df_stays['age'] != df_stays['anchor_age']).sum())
print('Largest difference, in years:', (df_stays['age'] - df_stays['anchor_age']).max())
df_stays[['anchor_age', 'age']].describe().round(1)

# 9. Is it a number the model can work with?
Cleaning makes the data complete and correct, and engineering adds what you know. The model still needs **numbers on a sensible scale**: every feature numeric, and carrying the knowledge you have about the problem.

## Encoding categories
*   Suppose we want a model of in-hospital death from age, admission type, gender and race
*   A model is arithmetic, $\hat{f}(X)$ from section 2, so like almost every model it needs **numbers**
*   `admission_type`, `gender` and `race` are text. **What could we do with them?**

### Ordinal encoding
*   **Ordinal encoding** gives each category an integer
*   For an ordered scale it is natural: pain of None, Mild, Moderate, Severe, Very severe as 0, 1, 2, 3, 4
*   It embeds two assumptions the model will use:
    *   **Order:** 4 is more than 3 is more than 2
    *   **Spacing:** Moderate is twice Mild

`pd.factorize` does it for any text column, numbering the categories alphabetically here. Race, encoded:

In [ ]:
codes, categories = pd.factorize(df_stays['race'], sort=True)
df_stays['race_code'] = codes

print(len(categories), 'categories')
code_table = df_stays[['race', 'race_code']].drop_duplicates()
code_table = code_table.sort_values('race_code')
code_table.head(8)

**» Investigate** — A model given `race_code` will treat a patient coded 28 as "more" than one coded 4, by 24 units. **What is category 28, what is category 4, and what does the model think the difference between them means?**

### One-hot encoding
*   **One-hot encoding** makes a new column for every category, `True` for the one that applies
*   No order, no spacing: the assumptions ordinal encoding embeds are gone
*   `pd.get_dummies` does it

### The two encodings, side by side
Four admissions, with `admission_type` encoded as one integer column (**ordinal**) or one 0/1 column per category (**one-hot**).

<div style="font-size:0.8em">

| `admission_type` | &nbsp; | ordinal | &nbsp; | elective | observation | urgent | emergency |
| :-- | :-- | --: | :-- | --: | --: | --: | --: |
| EMERGENCY | | 3 | | 0 | 0 | 0 | 1 |
| ELECTIVE | | 0 | | 1 | 0 | 0 | 0 |
| URGENT | | 2 | | 0 | 0 | 1 | 0 |
| OBSERVATION | | 1 | | 0 | 1 | 0 | 0 |

</div>

*   The integer column says emergency is more than urgent, by the same amount as urgent is more than observation
*   The 0/1 columns say only which category each admission is in

`race`, one-hot encoded, and how many stays fall in the rarest categories:

In [ ]:
race_onehot = pd.get_dummies(df_stays['race'])
print(race_onehot.shape)

race_counts = df_stays['race'].value_counts()
print((race_counts < 10).sum(), 'categories with fewer than 10 stays')
race_counts.tail(8)

**» Investigate** — 33 columns, 9 of them with fewer than 10 stays in 9,600. **What can a model learn about in-hospital death from a column that is `True` nine times?** Change `10` to `50`: how many columns are left that a model could learn anything from?

### Grouping before encoding
*   A category with a handful of rows is noise to a model, and a column the model can overfit to
*   The fix is a **judgement**, not a method: merge categories into groups that mean something for the question
*   MIMIC's `race` mixes race, ethnicity and country (`PORTUGUESE`, `SOUTH AMERICAN`); half the categories are subcategories (`WHITE - RUSSIAN`)
*   You will do this in the practice. Two cautions:
    *   Use an **established grouping** where one exists, e.g. the [NHS ethnic categories](https://www.datadictionary.nhs.uk/data_elements/ethnic_category.html), and say which
    *   Grouping race and ethnicity has consequences for who the model works for: **record what you merged, and why**

### Other encodings
*   For high-cardinality categories (hundreds of values, such as diagnosis codes), one-hot gives hundreds of columns
*   Alternatives: **target encoding**, **frequency encoding**, **binary encoding**, and learned embeddings
*   The reading list has references; none of them removes the grouping decision above

## Feature scaling
Six made-up patients, by heart rate and temperature. **Which patient is most like A?** The left panel draws both axes in raw units, at the same scale; the right panel puts each feature on the same scale first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Six made-up patients: heart rate in bpm, temperature in degrees C
names = ['A', 'B', 'C', 'D', 'E', 'F']
heart_rate = np.array([80, 81, 120, 86, 112, 95])
temperature = np.array([36.8, 39.6, 36.9, 36.7, 38.8, 37.2])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, x, y, title, xlabel, ylabel in [
        (axes[0], heart_rate, temperature, 'Raw units: both axes drawn to the same scale', 'Heart rate (bpm)', 'Temperature (°C)'),
        (axes[1], (heart_rate - heart_rate.mean()) / heart_rate.std(), (temperature - temperature.mean()) / temperature.std(),
         'Standardised: each feature has mean 0 and spread 1', 'Heart rate (standardised)', 'Temperature (standardised)')]:
    ax.scatter(x, y, s=60, color='#0000cd', zorder=3)
    for name, xi, yi in zip(names, x, y):
        ax.annotate(name, (xi, yi), xytext=(5, 5), textcoords='offset points', fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_aspect('equal', adjustable='datalim')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()


### Put every feature on the same scale
*   A model that measures how far apart two patients are, or multiplies each feature by a weight, sees the **left** panel: B's fever of 2.8 °C counts for less than D's extra 6 bpm, so B is A's nearest neighbour
*   So before such a model, every feature goes onto the same scale. Two recipes:
    *   **Normalisation**, or min-max scaling: squash the column into 0 to 1
    *   **Standardisation**: shift the column to a mean of 0 and a spread of 1, the z-score from the preparation notebook
*   scikit-learn does either in one line, and in Session 5 it learns the scale from the training rows only. The formulas are in the appendix

Min-max scaling of the maximum heart rate in the first 24 hours after admission, from the readings aggregated in section 6.

In [ ]:
hr_max_24h = df_first24h.groupby('hadm_id')['valuenum'].max()
hr_max_scaled = (hr_max_24h - hr_max_24h.min()) / (hr_max_24h.max() - hr_max_24h.min())

print('Largest maximum heart rate:', hr_max_24h.max())
print('Median after scaling:', round(hr_max_scaled.median(), 3))

**» Investigate** — Half the admissions have been scaled into the bottom 0.6% of the range. **Add a line before the scaling that keeps only plausible values**, `hr_max_24h = hr_max_24h[hr_max_24h <= 250]`, and run it again. How many values did that remove, and where is the median now?

Scaling is arithmetic on the whole column, so one error moves every value. **Clean, then scale.** And, as with imputation, the minimum and maximum are learnt from the data: in Session 5 they come from the training rows only.

# 10. Independent practice: a feature table for in-hospital death
From here to the **Regroup** is yours, in your pair: about 60 minutes. You will build **`df_features`**: one row per ICU stay, numeric columns, a `died` label, and nothing the model could not know at 24 hours.

* Each task is marked **» Modify** or **» Make**, and most ask for a **comment** recording a decision. The five questions from section 4 are the checklist for each feature, and the comments are what we collect at the Regroup.
* **Hint** blocks fold open: try first. The last cell of the preparation notebook is a one-page summary of the methods: keep it open.
* Everything builds on `df_stays` from section 1, in order.

Work together, and call me over when you get stuck. **Understanding beats finishing.**

## Part 1: One row per prediction
Some admissions have more than one ICU stay (815 in the cohort). A later stay already reflects what happened in the first, so the table keeps **the first ICU stay of each admission**.

**» Modify** — This cell keeps the **last** stay of each admission. Change it to keep the first, and check you have 8,647 rows.

In [ ]:
df_model = df_stays.sort_values('intime').drop_duplicates('hadm_id', keep='last')
print(len(df_model))

**» Modify** — The label balance in `df_model`. Change the cell so it gives **proportions, not counts**. What proportion of first stays end in death, and how does it compare with all stays in section 3?

In [ ]:
df_model['died'].value_counts()

**» Modify** — This cell is the box plot from section 3: the first-day maximum heart rate, split by `died`. Try **two other first-day vital signs**, one at a time, chosen from `heart_rate_mean`, `spo2_min`, `resp_rate_mean`, `sbp_min`, `glucose_max` and `temperature_mean`, and **write in the comment which of the three you have seen separates the groups most**: that is your first candidate.

In [ ]:
# Candidates:
df_model.boxplot(column='heart_rate_max', by='died')
plt.suptitle('')
plt.title('First-day value by in-hospital death')
plt.xlabel('Died in hospital')
plt.ylabel('heart_rate_max')
plt.show()

**» Make (stretch)** — **Test the differences.** Find a statistical test for a difference between two groups, apply it to the two vital signs you tried, and write in a comment: **which test you chose and why, and whether the p-values tell you which is the better feature.**

<details><summary><b>Hint</b></summary>

Split the column into the two groups first: `df_model[df_model['died'] == 1]['sbp_min'].dropna()` and the same with `== 0`. The tests you will find live in `scipy.stats`, and most take the two groups as their two arguments. With 8,647 rows, expect both to be significant; the question is what that is worth.

</details>

In [ ]:
# Test chosen, and why:
# Do the p-values tell you which is the better feature?

# Your code here

## Part 2: Features, one at a time
### Age
**» Make** — **A basic EDA of `age`**, which is already in `df_model` from section 8. At least: the **mean and median**, the **minimum and maximum**, the **interquartile range**, and a **histogram**. Write in the comment the one value that needs explaining. (MIMIC records everyone over 89 as 91 in their anchor year, so the oldest ages here are floors, not measurements.)

<details><summary><b>Hint</b></summary>

`.describe()` gives all five numbers at once (the IQR is the 75% row minus the 25% row), and `.plot.hist(bins=20)` the histogram, both from the preparation notebook; then `plt.xlabel`, `plt.ylabel` and `plt.show()`.

</details>

In [ ]:
# 1. Summary statistics of df_model['age']. The value that needs explaining:

# Your code here

In [ ]:
# 2. Histogram, with labelled axes

# Your code here

**» Make** — **In-hospital death rate by age band.** Make an `age_band` column that bins it at 40, 65 and 80, then group by it and take the mean of `died`. Then decide, in a comment: **age as a number, or as bands?** What does each let the model learn?

<details><summary><b>Hint</b></summary>

`pd.cut` bins a numeric column at the edges you give it, and `labels=` names the bins: `pd.cut(df_model['age'], [0, 40, 65, 80, 120], labels=['under 40', '40 to 64', '65 to 79', '80 and over'])`. Then `df_model.groupby('age_band', observed=True)['died'].mean()`; the mean of a 0/1 column is the proportion of 1s.

</details>

In [ ]:
# 1. Bin age into age_band
age_edges = [0, 40, 65, 80, 120]
age_labels = ['under 40', '40 to 64', '65 to 79', '80 and over']
df_model['age_band'] = ____

df_model['age_band'].value_counts()

In [ ]:
# 2. Death rate per band: group by age_band, mean of died. Number or bands, and why:

# Your code here

### Admission type
**» Make** — `admission_type` has nine values, several of them rare. **Group them into four** (`EMERGENCY`, `URGENT`, `OBSERVATION`, `ELECTIVE`) into a new column `admission_group`, then check the death rate in each group. Then **encode** `admission_group` as numbers, choosing ordinal or one-hot, and **write in the comment why**.

<details><summary><b>Hint</b></summary>

1. `df_model['admission_type'].value_counts()` to see the nine. `EW EMER.` and `DIRECT EMER.` are emergencies; `SURGICAL SAME DAY ADMISSION` is elective; the three `... OBSERVATION` types are observation.
2. `.replace({'EW EMER.': 'EMERGENCY', ...})` maps old values to new in one call.
3. Ordinal: `.map({'ELECTIVE': 0, 'OBSERVATION': 1, 'URGENT': 2, 'EMERGENCY': 3})`. One-hot: `pd.get_dummies(df_model['admission_group'], prefix='adm')`, then `pd.concat([df_model, dummies], axis=1)` to add the columns.

</details>

In [ ]:
# 1. The nine values
df_model['admission_type'].value_counts()

In [ ]:
# 2. Group them into four: complete the mapping, then apply it with .replace
grouping = {
    'EW EMER.': 'EMERGENCY',
    'DIRECT EMER.': ____,
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE',
    'SURGICAL SAME DAY ADMISSION': ____,
    'OBSERVATION ADMIT': 'OBSERVATION',
    'EU OBSERVATION': ____,
    'DIRECT OBSERVATION': ____,
    'AMBULATORY OBSERVATION': ____,
}
df_model['admission_group'] = ____

df_model['admission_group'].value_counts()

In [ ]:
# 3. Death rate in each group: group by admission_group, mean of died

# Your code here

In [ ]:
# 4. Encode admission_group as numbers. Ordinal or one-hot, and why:

# Your code here

### Race
The grouping below removes the ` - SUBCATEGORY` suffixes, merges the two spellings of Hispanic or Latino, and merges the four kinds of "not recorded". **Run it**, then look at what is left.

In [ ]:
race_grouped = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_grouped = race_grouped.str.replace(' OR ', '/')
race_grouped = race_grouped.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
df_model['race_group'] = race_grouped

df_model['race_group'].value_counts()

**» Make** — **One-hot encode `race_group`** into columns prefixed `race_`, add them to `df_model`, and print how many columns you made. Then, in a comment, **name one group you would still merge, and one you would not, and why**: 13 groups is fewer than 33, and still several have under 100 stays.

<details><summary><b>Hint</b></summary>

The same two lines as admission type: `pd.get_dummies(..., prefix='race')`, then `pd.concat([df_model, race_dummies], axis=1)`. `race_dummies.shape[1]` is the number of columns.

</details>

In [ ]:
# 1. One-hot encode race_group, prefix 'race', and add the columns to df_model

# Your code here

In [ ]:
# 2. How many columns did you make?
# Still merge:
# Keep separate:

# Your code here

### Cleaning the vital signs
**» Modify** — Section 7 found a maximum glucose of 999,999. This cell masks glucose values above 2,000. **Add a second line that does the same for `heart_rate_max`, at a limit you choose**, and write the limit and your reason in the comment. The largest `heart_rate_max` in the table is 277: error or extreme?

<details><summary><b>Hint</b></summary>

`df_model['heart_rate_max'].describe()` first. Then the same `.mask(condition)` pattern with your limit. There is no single right limit, but you must state the one you chose.

</details>

In [ ]:
# Heart rate limit, and why:
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)

print(df_model['glucose_max'].max())
print(df_model['heart_rate_max'].max())

### Filling the gaps
**» Modify** — About 280 stays have no first-day temperature, and section 7 showed they are twice as likely to end in death. This cell fills them with the **mean**, into a new column `temperature_imputed`, leaving `temperature_mean` as it was. Change it to fill with the **median**, and **add a column `temperature_missing`** that is `True` where there was a gap, so the model can see it.

<details><summary><b>Hint</b></summary>

`temperature_missing` comes from `.isna()` on `temperature_mean`, the original column with the gaps still in it.

</details>

In [ ]:
temperature = df_model['temperature_mean']
df_model['temperature_imputed'] = temperature.fillna(temperature.mean())

print(df_model['temperature_imputed'].isna().sum())

### A diagnosis feature
**» Make** — **A `diabetes` column: 1 if the admission has a diabetes diagnosis code, 0 if not.** ICD-9 diabetes codes start with `250`; ICD-10 codes start with `E10` or `E11`. Two steps: find the rows of `df_diagnoses` with a diabetes code, then mark the stays whose `hadm_id` appears in them. Check: about 30% of stays.

<details><summary><b>Hint</b></summary>

1. Session 3's AF pattern: `.str.startswith('E10')` is `True` for ICD-10 codes starting `E10`; two conditions combine with `|`, each in its own brackets.
2. `df_model['hadm_id'].isin(df_diabetes['hadm_id'])` is `True` for every stay whose admission has a code; `.astype(int)` turns `True` and `False` into 1 and 0.

</details>

In [ ]:
# 1. The rows of df_diagnoses with a diabetes code
is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = ____
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]

len(df_diabetes)

In [ ]:
# 2. A diabetes column: 1 where the stay's hadm_id is among the admissions with a code, 0 otherwise
df_model['diabetes'] = ____

df_model['diabetes'].mean()

## Part 3: Assemble the table
**» Make** — **Build `df_features`**: `died`, plus every feature you have made or chosen, all numeric, nothing the model could not know at 24 hours. Then:

1. Check `df_features.isna().sum()`: decide what to do with any gaps left, and say why in a comment.
2. Check `df_features.dtypes`: no text columns.
3. **Scale** the vital signs and age with min-max or standardisation, in a comment saying which and why.

<details><summary><b>Hint</b></summary>

1. The cell lists the columns in three groups; add the vital signs you chose in Part 1 to `numeric_features`, with `temperature_imputed` rather than `temperature_mean`, and use the names you gave your one-hot tables in Part 2 (`adm_dummies` and `race_dummies` here) in `made_features`.
2. `df_features.select_dtypes('bool')` finds columns to convert with `.astype(int)`.
3. The two formulas are in the appendix. Apply one to each column in turn, or with a `for` loop over a list of column names.

</details>

In [ ]:
# 1. Build df_features: died, plus every feature you made or chose, all numeric
# The label, the numeric features, and the columns you made in Part 2: add yours to each list
label = ['died']
numeric_features = ['age', 'heart_rate_max', ____]
made_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)

df_features = df_model[label + numeric_features + made_features].copy()
df_features.shape

In [ ]:
# 2. Check for gaps and for text columns. What I did with the remaining gaps, and why:

# Your code here

In [ ]:
# 3. Scale the vital signs and age. Scaling, and why:

# Your code here

**» Make (stretch)** — **A MEWS heart-rate score.** The [Modified Early Warning Score](https://www.mdcalc.com/calc/1875/modified-early-warning-score-mews-clinical-deterioration) gives heart rate 2 points below 40, 1 for 41 to 50, 0 for 51 to 100, 1 for 101 to 110, 2 for 111 to 129, and 3 from 130. Make `mews_hr` from `heart_rate_mean` with `pd.cut`. Would you give the model `mews_hr`, `heart_rate_mean`, or both?

<details><summary><b>Hint</b></summary>

`pd.cut(df_model['heart_rate_mean'], [0, 40, 50, 100, 110, 130, 1000], labels=[2, 1, 0, 1, 2, 3], ordered=False)`. The labels repeat, so `ordered=False` is needed. `.astype(int)` afterwards.

</details>

In [ ]:
# Your code here

**» Make (stretch)** — **Stays shorter than a day.** 201 stays in the cohort ended in death within 24 hours of ICU admission, and others were discharged within the day. At our prediction time their outcome was already known. **Remove the stays shorter than one day** (`los` is in days) and recompute the death rate. How much does the label change, and what did the short stays do to the first-day vital signs?

In [ ]:
# Your code here

**» Make (stretch)** — **A laboratory feature.** The `first_day_lab` view summarises the first day's blood tests. Load it, pick one test (`creatinine_max` or `wbc_max` are nearly complete; `albumin_min` is not), check how much is missing and how it relates to `died`, and add it to `df_features` with a stated decision about the gaps.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

# Your code here

## Regroup

Stop where you are: we come back together for the Closing.

If you didn't finish, finish after the session: Parts 2 and 3 are the ones Session 5 builds on, and the stretch tasks are optional.

Finished early? Help a neighbour.

# 11. Closing
## What your table depends on
Read out your comments. Then, in your pair for two minutes, and together:

1. **Let the model decide.** You chose two vital signs by eye from the box plots. A colleague says: give the model all 24 first-day columns and let it work out which matter. What do you say, and what from section 2 is on your side?
2. **Definition.** `died` is death in hospital. `dod` is recorded for about a third of the survivors as well: MIMIC follows patients for a year. A model of `dod IS NOT NULL` is a model of something else. Which would a surgeon planning an operation want?
3. **Nobody took a temperature.** Stays with no first-day temperature were twice as likely to end in death, and `temperature_missing` hands that to the model. What is the model actually learning about? Would this feature transfer well between settings with potentially different recording practices?

Session 5 trains models on a table like yours, and the first thing it does is split it in two.

## Wrap up
*   A model needs a feature table: one row per prediction, numeric columns, a label, and nothing from after the prediction time
*   Every feature has to pass five questions: known at prediction time, into the row, complete and correct, carrying what you know, a number the model can work with. Each method leaves a decision, and the decision comes from how the data were produced
*   Imputation, scaling and encoding all learn from the data: in Session 5 they learn from the training rows only
*   MIMIC data stay in your notebooks: do not share outputs, and do not paste rows into AI assistants

## Before Session 5
1. **Finish the feature table**, Parts 2 and 3, and check your decisions against the solutions notebook once it is released.
2. **Session 5 opens with a five-minute talk on model drift and the Epic sepsis model**, from the trio listed on [Canvas](https://canvas.imperial.ac.uk/courses/8306). Everyone else: while they talk, write down one question to ask them.
3. Reading list: chapters 12, 13 and 15 of *Secondary Analysis of Electronic Health Records*; pages 51 to 88 of *Hands-On Machine Learning with Scikit-Learn, Keras and TensorFlow*; the references on the curse of dimensionality and feature selection.
4. Coursework.

# Self-study appendices

## Nomenclature
Machine learning resources mix names borrowed from statistics and regression analysis. Strictly they are not identical; this course uses the red terms.

| **Role in model**     | **Machine learning**                       | **Statistics**            | **Regression analysis**    |
|------------------------|------------------------------------------|---------------------------|----------------------------|
| **Input variables**    | Predictors                               | Explanatory variables     | Regressors                 |
|                        | <span style="color:red">Features</span>  | Covariates                | Independent variables      |
|                        | Attributes, inputs                       |                           |                            |
| **Output variable**    | Target                                   | Response variable         | Dependent variable         |
|                        | <span style="color:red">Labels</span>    | Outcome                   |                            |
|                        | Output                                   | Criterion                 |                            |

## Feature selection
*   Every column is a dimension; more dimensions make the training data sparser, and a sparser table makes overfitting easier (the "curse of dimensionality")
*   **Feature selection** picks which features a model gets; **dimensionality reduction** builds a few new features that stand in for many
*   The EDA in section 3 was the first pass: a feature that does not separate the groups, or that is one of 33 near-empty columns, is a candidate to drop
*   References on both are in the reading list

## Significant is not the same as useful
Two tests on the feature-table question from section 3. A **chi-square test** asks whether two categorical variables are independent: here, gender and in-hospital death. The p-value is the probability of a split at least this uneven if they were.

In [ ]:
from scipy.stats import chi2_contingency

death_by_gender = pd.crosstab(df_stays['gender'], df_stays['died'], normalize='index')
chi2, p_value, dof, expected = chi2_contingency(pd.crosstab(df_stays['gender'], df_stays['died']))

print(death_by_gender.round(3))
print('p-value:', round(p_value, 3))

A **Mann-Whitney U test** compares a numeric variable between two groups without assuming a normal distribution: here, first-day mean temperature between the stays that ended in death and the rest.

In [ ]:
from scipy.stats import mannwhitneyu

temperature_died = df_stays[df_stays['died'] == 1]['temperature_mean'].dropna()
temperature_survived = df_stays[df_stays['died'] == 0]['temperature_mean'].dropna()
u_stat, p_value = mannwhitneyu(temperature_died, temperature_survived)

print('Median, died:    ', round(temperature_died.median(), 2))
print('Median, survived:', round(temperature_survived.median(), 2))
print('p-value:', p_value)

One test is far from significant, the other is highly significant, and neither tells you whether the feature is useful. Gender has no detectable relationship with the label and no separation. Temperature has a p-value near 10⁻¹⁰ and a difference of seven hundredths of a degree: with 9,600 rows almost any difference is surprising under the assumption of no relationship, and the model still has nothing to use. A p-value measures surprise, not size. For a feature, look at how far the groups separate, and at whether the column is known at prediction time.

## Scaling: the formulas, and scikit-learn
**Normalisation** (min-max scaling) squeezes a feature into 0 to 1:

$$ x_{normalised} = \frac{x - x_{min}}{x_{max} - x_{min}} $$

**Standardisation** centres it on 0 with a standard deviation of 1, which is the z-score:

$$ x_{standardised} = \frac{x - \mu_{x}}{\sigma_{x}} $$

`MinMaxScaler` and `StandardScaler` apply them. The difference from doing it by hand: `fit` learns the minimum and maximum (or mean and standard deviation) from one table, and `transform` applies them to any table, so the test set can be scaled with what was learnt from the training set.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_scaled = df_stays[['heart_rate_mean']].dropna()

minmax = MinMaxScaler()
df_scaled['normalised'] = minmax.fit_transform(df_scaled[['heart_rate_mean']])

standard = StandardScaler()
df_scaled['standardised'] = standard.fit_transform(df_scaled[['heart_rate_mean']])

df_scaled.describe().round(2)

## Transforming a distribution
A long right tail compresses most values into a small part of the range, for min-max scaling and for many models. A log transform pulls the tail in. Glucose, cleaned of the errors first:

In [ ]:
import numpy as np

glucose = df_stays['glucose_max']
glucose = glucose.mask(glucose > 2000)
glucose = glucose.dropna()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
glucose.plot.hist(bins=50, ax=axes[0], title='Maximum first-day glucose')
np.log(glucose).plot.hist(bins=50, ax=axes[1], title='Log of the same')
plt.show()

## More imputation methods
*   **Constant:** `df['A'].fillna(0)`, where the constant has a meaning (no readings means a count of zero)
*   **Average:** `df['B'].fillna(df['B'].median())`, mean or mode as the column suits
*   **Random (hot-deck):** draw substitutes from the observed values, which keeps the spread that a mean fill removes: `rnd = pd.Series(np.random.choice(df['C'].dropna(), size=len(df)))`, then `df['C'].fillna(rnd)`
*   **Regression:** predict the missing value from the other columns
*   **K-nearest neighbours:** take the value from the most similar rows; `sklearn.impute.KNNImputer`
*   **Indicator:** keep a `_missing` column alongside whichever fill you use

The more a method learns from the data, the more important it is to fit it on the training rows only.